In [3]:
import os
import requests
import json
from dotenv import load_dotenv
import requests_cache

In [5]:
"""
# load .env file, get API key
load_dotenv()
API_KEY = os.getenv("api_key")

# make sure API key is available
if not API_KEY:
    raise ValueError("no API key found")
"""
# set up a cached session
session = requests_cache.CachedSession('output/steam_api_cache', expire_after=86400)

In [6]:
# sample output
def fetch_steam_app_data(app_id: int) -> dict:
    """
    Fetch detailed information about a Steam application using the Storefront API.
    """
    
    # define the target API endpoint
    url = "https://store.steampowered.com/api/appdetails"
    
    # construct query parameters for the GET request
    params = {
        "appids": app_id,
        "key": API_KEY,
        "cc": "us",      # currency: USD
        "l": "english"   # language: English
    }
    
    try:

        # use cached session to sent request, timeout after 10 seconds
        response = session.get(url, params=params, timeout=10)
        response.raise_for_status()
        
        # parse the JSON response into a json
        raw_data = response.json()
        
        # check if the response contains valid data for the given app_id
        app_id_str = str(app_id)
        if app_id_str in raw_data and raw_data[app_id_str].get("success"):
            return raw_data[app_id_str]["data"]
        else:
            print(f"No valid data found for AppId:{app_id_str}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"HTTP Request failed: {e}")
        return None


In [7]:
sample_output = fetch_steam_app_data(1091500)

with open("output/sample_app_data.json", "w") as f:
    json.dump(sample_output, f, indent=4)